# Logic Laboratory — Logical Reasoning for Planning

**Artificial Intelligence — Using an LLM to Construct and Test a Simple Planning Agent**

Central idea: **Logic + Search = Planning**.

A planning problem is $(I, A, G)$ — initial state, actions, goal. Each action has
**preconditions** (what must hold to apply it) and **effects** (what becomes true/false after).
Logically, action $a$ is applicable in state $S$ when $S \models \text{Preconditions}(a)$.

This notebook covers every task in `Logic.pdf`:

* **Task 0** — understand the planning problem.
* **Task 1** — construct a plan by hand (with state trace).
* **Task 2** — implement a STRIPS-style planner (BFS) in Python.
* **Task 3** — test it (solvable / impossible / irrelevant-action cases).
* **Task 4** — explain how logic and search cooperate.
* **Task 5** — generated-explanation vs independent verification.
* **Tasks 6–8 (optional)** — Prolog as an independent logical verifier.


## The Scenario

A warehouse robot must deliver a package. Three locations: **A, B, C**. The robot and package
both start at **A**; the package must end at **C**.

$$I = \{\text{At(Robot,A)},\ \text{At(Package,A)}\}, \qquad G = \{\text{At(Package,C)}\}.$$

Actions:

* **Move(x, y):** pre `At(Robot,x)`; effects `¬At(Robot,x)`, `At(Robot,y)`. Connected pairs:
  A–B, B–A, B–C, C–B.
* **PickUp(Package, x):** pre `At(Robot,x)`, `At(Package,x)`; effects `¬At(Package,x)`,
  `Holding(Package)`.
* **Drop(Package, x):** pre `At(Robot,x)`, `Holding(Package)`; effects `¬Holding(Package)`,
  `At(Package,x)`.


## Task 0 — Understand the Planning Problem

**(a) Initial state $I$:** `{At(Robot,A), At(Package,A)}`.
**(b) Goal $G$:** `{At(Package,C)}`.
**(c) Actions:** Move, PickUp, Drop (parameterised by locations).
**(d) Preconditions / effects:** as listed above.

**Which actions are applicable in $I$?**

* `PickUp(Package, A)` — preconditions `At(Robot,A)` ∧ `At(Package,A)`. Both hold in $I$, so it
  **is applicable**. ✓
* `Drop(Package, C)` — preconditions `At(Robot,C)` ∧ `Holding(Package)`. Neither holds in $I$
  (the robot is at A and holds nothing), so it is **not applicable**. ✗

> **Think About It:** an action in the list is only *applicable* when **all** its preconditions
> are satisfied by the current state. Checking that is exactly the logical test
> $S \models \text{Preconditions}(a)$ — this is where logical reasoning enters planning.


## Task 1 — Construct a Plan by Hand

A valid plan is $a_1,\dots,a_n$ with $I \xrightarrow{a_1} S_1 \to \cdots \to S_n$ and
$S_n \models G$. One correct plan:

$$\text{PickUp(Package,A)},\ \text{Move(A,B)},\ \text{Move(B,C)},\ \text{Drop(Package,C)}.$$

| State | Facts |
|---|---|
| $S_0=I$ | At(Robot,A), At(Package,A) |
| $S_1$ (after PickUp(Package,A)) | At(Robot,A), Holding(Package) |
| $S_2$ (after Move(A,B)) | At(Robot,B), Holding(Package) |
| $S_3$ (after Move(B,C)) | At(Robot,C), Holding(Package) |
| $S_4$ (after Drop(Package,C)) | At(Robot,C), At(Package,C) |

$S_4 \models G$ since `At(Package,C)` holds. (Picking the package up at A and carrying it is
necessary — simply moving the *robot* to C would leave the package behind at A.)


## Task 2 — Implement the Planner

Specification (as would be given to an LLM): a state is a **set of logical propositions**; each
action has a name, positive/negative preconditions, and positive/negative effects. An action is
applicable iff its positive preconditions are all present **and** its negative preconditions are
all absent. Applying it removes negative effects and adds positive effects. Use **breadth-first
search** to find a shortest plan; detect when no plan exists; print the plan and the state after
each action.


In [1]:
from collections import deque
from dataclasses import dataclass, field

@dataclass(frozen=True)
class Action:
    name: str
    pre_pos: frozenset = field(default_factory=frozenset)  # must be present
    pre_neg: frozenset = field(default_factory=frozenset)  # must be absent
    eff_pos: frozenset = field(default_factory=frozenset)  # added
    eff_neg: frozenset = field(default_factory=frozenset)  # removed

    def applicable(self, state):
        """S |= Preconditions(a): all positive preconditions in, all negative ones out."""
        return self.pre_pos <= state and self.pre_neg.isdisjoint(state)

    def apply(self, state):
        """Apply(S, a): remove negative effects, then add positive effects."""
        return (state - self.eff_neg) | self.eff_pos


def bfs_plan(initial, goal, actions):
    """Breadth-first STRIPS planner.

    initial, goal : frozenset of proposition strings (goal = required facts).
    Returns (plan, trace) or (None, None) if no plan exists.
    """
    start = frozenset(initial)
    goal = frozenset(goal)
    if goal <= start:
        return [], [start]

    frontier = deque([start])
    came_from = {start: (None, None)}  # state -> (parent state, action)
    while frontier:
        state = frontier.popleft()
        for a in actions:
            if a.applicable(state):
                nxt = a.apply(state)
                if nxt not in came_from:
                    came_from[nxt] = (state, a)
                    if goal <= nxt:                 # goal test: G subset of state
                        # reconstruct
                        plan, trace, node = [], [nxt], nxt
                        while came_from[node][0] is not None:
                            parent, act = came_from[node]
                            plan.append(act)
                            trace.append(parent)
                            node = parent
                        plan.reverse(); trace.reverse()
                        return plan, trace
                    frontier.append(nxt)
    return None, None


In [2]:
def warehouse_actions(include_pickup=True, extra_moves=None):
    """Build the warehouse action set. Connected location pairs -> Move actions."""
    connections = [("A", "B"), ("B", "A"), ("B", "C"), ("C", "B")]
    if extra_moves:
        connections = connections + extra_moves
    actions = []
    for x, y in connections:
        actions.append(Action(
            name=f"Move({x},{y})",
            pre_pos=frozenset({f"At(Robot,{x})"}),
            eff_pos=frozenset({f"At(Robot,{y})"}),
            eff_neg=frozenset({f"At(Robot,{x})"}),
        ))
    for loc in ("A", "B", "C"):
        if include_pickup:
            actions.append(Action(
                name=f"PickUp(Package,{loc})",
                pre_pos=frozenset({f"At(Robot,{loc})", f"At(Package,{loc})"}),
                eff_pos=frozenset({"Holding(Package)"}),
                eff_neg=frozenset({f"At(Package,{loc})"}),
            ))
        actions.append(Action(
            name=f"Drop(Package,{loc})",
            pre_pos=frozenset({f"At(Robot,{loc})", "Holding(Package)"}),
            eff_pos=frozenset({f"At(Package,{loc})"}),
            eff_neg=frozenset({"Holding(Package)"}),
        ))
    return actions


def show(plan, trace):
    if plan is None:
        print("No plan found")
        return
    print(f"Plan ({len(plan)} actions):")
    for i, a in enumerate(plan, 1):
        print(f"  {i}. {a.name}")
    print("\nState trace:")
    for i, s in enumerate(trace):
        facts = ", ".join(sorted(s))
        label = "S0 (initial)" if i == 0 else f"S{i} after {plan[i-1].name}"
        print(f"  {label:<28}: {{{facts}}}")


## Task 3 — Test the Generated Planner

### Test A — solvable (original problem)


In [3]:
I = frozenset({"At(Robot,A)", "At(Package,A)"})
G = frozenset({"At(Package,C)"})

plan, trace = bfs_plan(I, G, warehouse_actions())
show(plan, trace)

# Independently verify every action's preconditions hold where it is executed.
def verify_plan(initial, goal, plan):
    state = frozenset(initial)
    for a in plan:
        assert a.applicable(state), f"Precondition fails before {a.name}: {sorted(state)}"
        state = a.apply(state)
    assert frozenset(goal) <= state, f"Goal not reached; final state {sorted(state)}"
    return True

print("\nIndependent verification:",
      "VALID" if verify_plan(I, G, plan) else "INVALID")


Plan (4 actions):
  1. PickUp(Package,A)
  2. Move(A,B)
  3. Move(B,C)
  4. Drop(Package,C)

State trace:
  S0 (initial)                : {At(Package,A), At(Robot,A)}
  S1 after PickUp(Package,A)  : {At(Robot,A), Holding(Package)}
  S2 after Move(A,B)          : {At(Robot,B), Holding(Package)}
  S3 after Move(B,C)          : {At(Robot,C), Holding(Package)}
  S4 after Drop(Package,C)    : {At(Package,C), At(Robot,C)}

Independent verification: VALID


### Test B — impossible (remove the PickUp action)

Without `PickUp`, the robot can never hold the package, so `At(Package,C)` is unreachable. The
planner must report **No plan found**, not invent an action.


In [4]:
plan_b, trace_b = bfs_plan(I, G, warehouse_actions(include_pickup=False))
show(plan_b, trace_b)
print("\nExpected: No plan found (goal unreachable without PickUp).")


No plan found

Expected: No plan found (goal unreachable without PickUp).


### Test C — irrelevant actions

Moving the *robot* to C does **not** achieve `At(Package,C)`. We set the goal to the robot
reaching C and confirm the planner distinguishes the robot's location from the package's. Moving
the robot around while the package stays at A must never be mistaken for delivering the package.


In [5]:
# C1: goal = robot at C. Plan should move the robot only, and the package stays at A.
G_robot = frozenset({"At(Robot,C)"})
plan_c, trace_c = bfs_plan(I, G_robot, warehouse_actions())
show(plan_c, trace_c)
final_state = trace_c[-1]
print("\nPackage still at A in final state?",
      "At(Package,A)" in final_state, "(robot reaching C != package at C)")

# C2: confirm the package-delivery goal is NOT satisfied by that robot-only plan.
print("Package-delivery goal satisfied by robot-only plan?",
      frozenset({"At(Package,C)"}) <= final_state, "(expected False)")


Plan (2 actions):
  1. Move(A,B)
  2. Move(B,C)

State trace:
  S0 (initial)                : {At(Package,A), At(Robot,A)}
  S1 after Move(A,B)          : {At(Package,A), At(Robot,B)}
  S2 after Move(B,C)          : {At(Package,A), At(Robot,C)}

Package still at A in final state? True (robot reaching C != package at C)
Package-delivery goal satisfied by robot-only plan? False (expected False)


### Recorded results

| Test | Initial | Goal | Plan found? | Plan | Valid? |
|---|---|---|---|---|---|
| A solvable | At(Robot,A),At(Package,A) | At(Package,C) | Yes | PickUp(A), Move(A,B), Move(B,C), Drop(C) | Yes (verified) |
| B impossible | same, no PickUp action | At(Package,C) | **No** | — | correctly reports failure |
| C irrelevant | At(Robot,A),At(Package,A) | At(Robot,C) | Yes | Move(A,B), Move(B,C) | package stays at A; delivery goal still **False** |


## Task 4 — Logic and Search

The planner combines two distinct ideas:

* **Logical reasoning** decides, for a state $S$ and action $a$, whether
  $S \models \text{Preconditions}(a)$ — implemented by `Action.applicable`. If so, the successor
  is $S' = \text{Apply}(S,a)$ — implemented by `Action.apply` (delete negative effects, add
  positive effects).
* **Search** decides *which* sequence of applicable actions to explore — implemented by the BFS
  loop over successor states.

Completing the pipeline (the missing step `?` is **apply the action / generate the successor
state**):

```
Current state
   |
Check action preconditions        <-- logical reasoning: S |= Pre(a)?
   |
Apply the action (S' = Apply(S,a)) <-- the missing step
   |
Generate successor state
   |
Search over alternatives          <-- search: which successors to expand (BFS)
   |
Goal?                              <-- logical test: G subset of S'
```

In words: **logic determines what is *possible*** (which actions may fire and how they transform
the world); **search determines what to *try*** (which of the possible action sequences leads to
the goal). Together they turn a set of action schemas into a plan.


## Task 5 — Generated Explanation vs Independent Verification

An LLM can *explain* why its plan is valid, but that explanation is just more generated text. Our
`verify_plan` function re-executes the plan against the formal action model and checks every
precondition independently — shown below as a per-step audit.


In [6]:
def audit(initial, plan):
    """Print, for each action, the state before it and confirm its preconditions hold."""
    state = frozenset(initial)
    print(f"S0: {{{', '.join(sorted(state))}}}")
    for i, a in enumerate(plan, 1):
        ok = a.applicable(state)
        need = ", ".join(sorted(a.pre_pos))
        print(f"  apply {a.name}: preconditions {{{need}}} satisfied? {ok}")
        state = a.apply(state)
        print(f"S{i}: {{{', '.join(sorted(state))}}}")
    return state

audit(I, plan)
print("\nWhich to trust: the independently executed state transitions (b).")
print("A generated explanation can be fluent yet wrong; the executed transitions are")
print("computed from the formal model, so they are the authoritative check.")


S0: {At(Package,A), At(Robot,A)}
  apply PickUp(Package,A): preconditions {At(Package,A), At(Robot,A)} satisfied? True
S1: {At(Robot,A), Holding(Package)}
  apply Move(A,B): preconditions {At(Robot,A)} satisfied? True
S2: {At(Robot,B), Holding(Package)}
  apply Move(B,C): preconditions {At(Robot,B)} satisfied? True
S3: {At(Robot,C), Holding(Package)}
  apply Drop(Package,C): preconditions {At(Robot,C), Holding(Package)} satisfied? True
S4: {At(Package,C), At(Robot,C)}

Which to trust: the independently executed state transitions (b).
A generated explanation can be fluent yet wrong; the executed transitions are
computed from the formal model, so they are the authoritative check.


**Which should you trust more — (a) the LLM's explanation, or (b) the independently executed
state transitions?** The transitions (b). The explanation is produced by the same kind of system
that produced the plan and can be confidently mistaken. Re-executing the plan against the formal
precondition/effect model is an *independent* check grounded in the problem definition.

> **Principle:** *A generated explanation is not the same as an independent verification.*


## Optional Extension — Prolog as a Logical Verifier

Prolog expresses knowledge as **facts + rules**, and answers **queries** by inference. The lab's
`planner.pl` would contain:

```prolog
connected(a,b).
connected(b,a).
connected(b,c).
connected(c,b).

can_move(X,Y) :- connected(X,Y).
valid_move(X,Y) :- connected(X,Y).
```

Queries and expected answers:

* `?- can_move(a,b).` → **true** (the fact `connected(a,b)` exists, so the rule body succeeds).
* `?- can_move(a,c).` → **false** (there is no `connected(a,c)` fact and no rule derives it; A
  connects only to B).

**(c)** The rule `can_move(X,Y) :- connected(X,Y).` is Prolog's way of writing the implication
$\text{Connected}(X,Y) \rightarrow \text{CanMove}(X,Y)$: *if* `connected(X,Y)` can be proved,
*then* `can_move(X,Y)` holds.

Below we reproduce the same **Facts + Rules → Inference** behaviour in pure Python so the notebook
is self-contained (no SWI-Prolog install needed). The logic is identical to the Prolog program.


In [7]:
# A tiny Prolog-style engine: facts + rules, answered by backward chaining.
CONNECTED = {("a", "b"), ("b", "a"), ("b", "c"), ("c", "b")}

def connected(x, y):
    return (x, y) in CONNECTED

def can_move(x, y):       # can_move(X,Y) :- connected(X,Y).
    return connected(x, y)

def valid_move(x, y):     # valid_move(X,Y) :- connected(X,Y).
    return connected(x, y)

print("Task 6 queries:")
print("  can_move(a,b) ->", can_move("a", "b"), "(expected true)")
print("  can_move(a,c) ->", can_move("a", "c"), "(expected false)")

print("\nTask 7 — check the Python-planner's proposed moves Move(a,b), Move(b,c):")
for x, y in [("a", "b"), ("b", "c"), ("a", "c")]:
    print(f"  valid_move({x},{y}) ->", valid_move(x, y))

print("\nChallenge — is Move(a,c) supported by the warehouse knowledge?")
print("  valid_move(a,c) ->", valid_move("a", "c"),
      "=> NOT supported; the robot cannot move A->C directly.")


Task 6 queries:
  can_move(a,b) -> True (expected true)
  can_move(a,c) -> False (expected false)

Task 7 — check the Python-planner's proposed moves Move(a,b), Move(b,c):
  valid_move(a,b) -> True
  valid_move(b,c) -> True
  valid_move(a,c) -> False

Challenge — is Move(a,c) supported by the warehouse knowledge?
  valid_move(a,c) -> False => NOT supported; the robot cannot move A->C directly.


### Task 8 — Chained inference

Prolog program:

```prolog
wet_road.
slippery    :- wet_road.
reduce_speed :- slippery.
?- reduce_speed.
```

The query `reduce_speed` **succeeds**. Reasoning as a chain of implications:

$$\text{wet\_road} \ \Rightarrow\ \text{slippery} \ \Rightarrow\ \text{reduce\_speed}.$$

(Fact ⇒ Rule ⇒ Rule ⇒ Conclusion.) Reproduced in Python:


In [8]:
wet_road = True
slippery = wet_road           # slippery :- wet_road.
reduce_speed = slippery       # reduce_speed :- slippery.
print("reduce_speed ->", reduce_speed, "(succeeds)")
print("Chain:  wet_road => slippery => reduce_speed")


reduce_speed -> True (succeeds)
Chain:  wet_road => slippery => reduce_speed


### Reflection on Prolog & the Takeaway

1. **Fact vs rule:** a *fact* asserts something unconditionally (`connected(a,b).`); a *rule*
   derives a head when its body can be proved (`can_move(X,Y) :- connected(X,Y).`).
2. **A query** asks whether a goal *follows from* the knowledge base — i.e. whether it can be
   proved from the facts and rules by inference.
3. **Why use Prolog to verify a Python plan?** It is an **independent** reasoning system working
   from a declarative description of the world, so it can confirm or refute a candidate action
   without relying on the code (or LLM) that produced it.
4. **Advantage of an independent verifier** when the plan came from an LLM: the generator and the
   checker are different mechanisms, so an error introduced during generation is unlikely to be
   silently reproduced by the checker.

> **Key idea:** an AI system can *generate* a candidate solution, while a separate logical system
> *checks* it.  More generally:
> **Understand → Specify → Generate → Execute → Verify**, and **Logic + Search = Planning.**
